### Traitement des données : Dans cette partie, il s'agira pour nous de préparer les données pour la modélisation à travers le nettoyage, l'encodage et le feature engineering

In [18]:
# Importating necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

In [19]:
# Charging the  dataset
data = pd.read_csv("../data/raw/test.csv")
data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


In [20]:
# Types of columns in the dataset
data.dtypes

Id                 int64
MSSubClass         int64
MSZoning          object
LotFrontage      float64
LotArea            int64
                  ...   
MiscVal            int64
MoSold             int64
YrSold             int64
SaleType          object
SaleCondition     object
Length: 80, dtype: object

In [21]:
# Checking missing values
data.isnull().sum()

Id                 0
MSSubClass         0
MSZoning           4
LotFrontage      227
LotArea            0
                ... 
MiscVal            0
MoSold             0
YrSold             0
SaleType           1
SaleCondition      0
Length: 80, dtype: int64

In [22]:
# Visualization of the first 3 rows of the dataset after removing the 'customer_id' column
data.head(3)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal


In [23]:
# Checking for duplicates
print(data.duplicated().sum())

0


In [24]:
# Checking categorial columns and let's look at the unique values
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", categorical_cols)

Categorical columns: ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']


In [25]:
# Let's drop the features with more than 50% missing values
threshold = 0.5
missing_percentages = data.isnull().mean()
columns_to_drop = missing_percentages[missing_percentages > threshold].index
data.drop(columns=columns_to_drop, inplace=True)
print(f"Dropped columns: {columns_to_drop.tolist()}")
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()

Dropped columns: ['Alley', 'MasVnrType', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature']


In [26]:
# Let's drop again features with little impact on the target variable
data = data.drop(columns=['BsmtFinSF2', 'BsmtHalfBath', 'MiscVal', 'MoSold'])
data.head(3)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,Reg,Lvl,AllPub,Inside,...,Y,140,0,0,0,120,0,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,IR1,Lvl,AllPub,Corner,...,Y,393,36,0,0,0,0,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,IR1,Lvl,AllPub,Inside,...,Y,212,34,0,0,0,0,2010,WD,Normal


In [27]:
# Multicollinearity correction
data = data.drop(columns=['GarageCars', 'TotRmsAbvGrd'])

In [28]:
# Imputing missing values for numerical columns with the median
numerical_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
for col in numerical_cols:
    data[col] = data[col].fillna(data[col].median())

In [29]:
# Imputing missing values for categorical columns with the mode
for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

In [30]:
# Encoding categorical variables using One-Hot Encoding
encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
encoded_cols = encoder.fit_transform(data[categorical_cols])
encoded_col_names = encoder.get_feature_names_out(categorical_cols)
encoded_df = pd.DataFrame(encoded_cols, columns=encoded_col_names, index=data.index)
data = pd.concat([data.drop(columns=categorical_cols), encoded_df], axis=1)

In [31]:
# Feature engineering

# Surface totale de la maison
data['TotalSF'] = data['TotalBsmtSF'] + data['1stFlrSF'] + data['2ndFlrSF']

# Age de la maison au moment de la vente
data['HouseAge'] = data['YrSold'] - data['YearBuilt']

# Années depuis la rénovation
data['YearsSinceRemodel'] = data['YrSold'] - data['YearRemodAdd']

# Surface totale des terrasses/porches
data['TotalPorchSF'] = data['OpenPorchSF'] + data['EnclosedPorch'] + data['ScreenPorch'] + data['3SsnPorch'] + data['WoodDeckSF']

# Nombre total de salles de bains
data['TotalBathrooms'] = data['FullBath'] + data['HalfBath'] * 0.5 + data['BsmtFullBath']

# La maison a-t-elle un garage ?
data['HasGarage'] = (data['GarageArea'] > 0).astype(int)

# La maison a-t-elle une piscine ?
data['HasPool'] = (data['PoolArea'] > 0).astype(int)

# La maison a-t-elle une cheminée ?
data['HasFireplace'] = (data['Fireplaces'] > 0).astype(int)

# La maison a-t-elle été rénovée ?
data['IsRemodeled'] = (data['YearBuilt'] != data['YearRemodAdd']).astype(int)

In [32]:
# Let's drop the original columns that we used to create the new features
data = data.drop(columns=[
    '1stFlrSF', '2ndFlrSF', 'TotalBsmtSF', 'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 
    '3SsnPorch', 'WoodDeckSF', 'FullBath', 'HalfBath', 'BsmtFullBath',     'YearBuilt', 'YearRemodAdd'                 
])

In [33]:
# Visualization of the first 3 rows of the dataset after processing
data.head()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,MasVnrArea,BsmtFinSF1,BsmtUnfSF,LowQualFinSF,...,SaleCondition_Partial,TotalSF,HouseAge,YearsSinceRemodel,TotalPorchSF,TotalBathrooms,HasGarage,HasPool,HasFireplace,IsRemodeled
0,1461,20,80.0,11622,5,6,0.0,468.0,270.0,0,...,0.0,1778.0,49,49,260,1.0,1,0,0,0
1,1462,20,81.0,14267,6,6,108.0,923.0,406.0,0,...,0.0,2658.0,52,52,429,1.5,1,0,0,0
2,1463,60,74.0,13830,5,5,0.0,791.0,137.0,0,...,0.0,2557.0,13,12,246,2.5,1,0,1,1
3,1464,60,78.0,9978,6,6,20.0,602.0,324.0,0,...,0.0,2530.0,12,12,396,2.5,1,0,1,0
4,1465,120,43.0,5005,8,5,0.0,263.0,1017.0,0,...,0.0,2560.0,18,18,226,2.0,1,0,0,0


In [34]:
# Verify missing values
print(data.isnull().sum())

Id                0
MSSubClass        0
LotFrontage       0
LotArea           0
OverallQual       0
                 ..
TotalBathrooms    0
HasGarage         0
HasPool           0
HasFireplace      0
IsRemodeled       0
Length: 204, dtype: int64


In [35]:
# Save test
data.to_csv("../data/processed/test_clean.csv", index=False)
print(f"Shape finale: {data.shape}")
print("Saved!")

Shape finale: (1459, 204)
Saved!
